In [2]:
import polars as pl
import json
from typing import List, Dict

In [3]:
interactions_output_parquet_path = '/home/jovyan/IRec/sigir/yambda_data/yambda_sequential_50m_filtered_reindexed.parquet'
df = pl.read_parquet(interactions_output_parquet_path)

In [12]:
def merge_and_save(parts_to_merge, dirr, output_name):
    merged = {}
    print(f"Merging {len(parts_to_merge)} files into {output_name}...")
    
    for part in parts_to_merge:
        # with open(fp, 'r') as f:
        #     part = json.load(f)
        for uid, items in part.items():
            if uid not in merged:
                merged[uid] = []
            merged[uid].extend(items)
            
    out_path = f"{dirr}/{output_name}"
    with open(out_path, 'w') as f:
        json.dump(merged, f)
    print(f"✓ Done: {out_path} (Users: {len(merged)})")


def merge_and_save_with_filter(parts_to_merge, dirr, output_name, min_history_len=5):
    merged = {}
    print(f"Merging {len(parts_to_merge)} files into {output_name} (min len={min_history_len})...")
    
    for part in parts_to_merge:
        for uid, items in part.items():
            if uid not in merged:
                merged[uid] = []
            merged[uid].extend(items)

    filtered_merged = {}
    filtered_count = 0
    
    for uid, items in merged.items():
        if len(items) >= min_history_len:
            filtered_merged[uid] = items
        else:
            filtered_count += 1
    
    print(f"Filtered {filtered_count} users with history < {min_history_len}")
    print(f"Remaining: {len(filtered_merged)} users")
    
    out_path = f"{dirr}/{output_name}"
    with open(out_path, 'w') as f:
        json.dump(filtered_merged, f)
    print(f"Done: {out_path} (Users: {len(filtered_merged)})")

In [4]:
def split_session_by_timestamps(
    df: pl.DataFrame,
    time_cutoffs: List[int],
    output_dir: str = None,
    return_dicts: bool = True
) -> List[Dict[int, List[int]]]:
    """
    Args:
        df: Polars DataFrame с колонками uid, item_ids (list), timestamps (list)
        time_cutoffs: Лист временных точек для разбиения
        output_dir: Директория для сохранения JSON файлов (опционально)
        return_dicts: Возвращать ли словари (как json_data format)
    
    Возвращает лист словарей в формате {user_id: [item_ids для интервала]}
    """
    
    result_dicts = []
    
    def extract_interval(df_source, start, end=None):
        q = df_source.lazy()
        q = q.explode(["item_ids", "timestamps"])
        
        if end is not None:
            q = q.filter(
                (pl.col("timestamps") >= start) & 
                (pl.col("timestamps") < end)
            )
        else:
            q = q.filter(
                pl.col("timestamps") >= start
            )
            
        q = q.group_by("uid").agg([
            pl.col("item_ids").alias("item_ids")
        ]).sort("uid")
        
        return q.collect()
    
    intervals = []
    current_start = 0
    for cutoff in time_cutoffs:
        intervals.append((current_start, cutoff))
        current_start = cutoff

    intervals.append((current_start, None))

    for start, end in intervals:
        subset = extract_interval(df, start, end)

        json_dict = {}
        for user_id, item_ids in subset.iter_rows():
            json_dict[user_id] = item_ids
        
        result_dicts.append(json_dict)

        if output_dir:
            if end is not None:
                filename = f"inter_new_[{start}_{end}).json"
            else:
                filename = f"inter_new_[{start}_inf).json"
            
            filepath = f"{output_dir}/{filename}"
            with open(filepath, 'w') as f:
                json.dump(json_dict, f, indent=2)
            
            print(f"✓ Сохранено: {filepath}")
    
    return result_dicts

In [4]:
df.head()

uid,timestamps,item_ids
u32,list[u32],list[u32]
600,"[1329190, 1329405, … 25997540]","[252026, 58171, … 201909]"
800,"[121100, 121290, … 25977310]","[20844, 198210, … 60455]"
1000,"[11335730, 11335925, … 25972225]","[46643, 57592, … 95670]"
1400,"[280570, 280735, … 25993315]","[4634, 213798, … 104891]"
1600,"[899275, 930305, … 25941890]","[223933, 154424, … 104876]"


# QUANTILE CUTOFF

In [7]:
def get_quantile_cutoffs(df, num_parts=4, base_ratio=None):
    """
    Считает cutoffs так, чтобы разбить данные на части.
    
    Args:
        num_parts: На сколько частей делить "хвост" истории.
        base_ratio: Какую долю данных отдать в Base (самую первую часть). 
                    Если None, делит всё поровну.
    """
    # Достаем все таймстемпы в один плоский массив
    # Это может занять память, если данных очень много (>100M), но для Beauty (2M) это ок
    all_ts = df.select(pl.col("timestamps").explode()).to_series().sort()
    total_events = len(all_ts)
    
    print(f"Всего событий: {total_events}")
    
    cutoffs = []
    
    if base_ratio:
        # Base занимает X% (например 80%), а остаток делим поровну на 3 части (Valid, Gap, Test)
        # Остаток = 1 - base_ratio
        # Каждая малая часть = (1 - base_ratio) / num_parts_tail
        
        base_idx = int(total_events * base_ratio)
        cutoffs.append(all_ts[base_idx]) # Первый cutoff отделяет Base
        
        remaining_events = total_events - base_idx
        part_size = remaining_events // num_parts # Делим остаток на 3 части (P1, P2, P3)
        
        current_idx = base_idx
        for _ in range(num_parts-1): # Нам нужно еще 2 границы, чтобы получить 3 части
            current_idx += part_size
            cutoffs.append(all_ts[current_idx])
            
    else:
        # Сценарий: Просто делим всё на N равных частей
        step = total_events // num_parts
        for i in range(1, num_parts):
            idx = i * step
            cutoffs.append(all_ts[idx])
            
    return cutoffs


In [8]:
equal_event_cutoffs = get_quantile_cutoffs(df, num_parts=4, base_ratio=0.8)

print("\n--- Новые Cutoffs (по количеству событий) ---")
print(f"Cutoffs: {equal_event_cutoffs}")

# Проверка распределения
intervals_eq = [0] + equal_event_cutoffs + [None]
print(intervals_eq)

Всего событий: 7371990

--- Новые Cutoffs (по количеству событий) ---
Cutoffs: [22138015, 23136375, 24137410, 25093085]
[0, 22138015, 23136375, 24137410, 25093085, None]


In [9]:
new_split_files = split_session_by_timestamps(
    df, 
    [22138015, 24137410, 25093085], 
    output_dir="/home/jovyan/IRec/data/Yambda/updated_quantile_splits/raw"
)

names = ["Base", "Gap", "Valid", "Test"]
for i, d in enumerate(new_split_files):
    num_users = len(d)
    
    num_events = sum(len(items) for items in d.values())
    
    print(f"{i:<10} {names[i]:<10} {num_users:<10} {num_events:<10}")

✓ Сохранено: /home/jovyan/IRec/data/Yambda/updated_quantile_splits/raw/inter_new_[0_22138015).json
✓ Сохранено: /home/jovyan/IRec/data/Yambda/updated_quantile_splits/raw/inter_new_[22138015_24137410).json
✓ Сохранено: /home/jovyan/IRec/data/Yambda/updated_quantile_splits/raw/inter_new_[24137410_25093085).json
✓ Сохранено: /home/jovyan/IRec/data/Yambda/updated_quantile_splits/raw/inter_new_[25093085_inf).json
0          Base       3813       5897592   
1          Gap        3315       737198    
2          Valid      3120       368599    
3          Test       3154       368601    


In [34]:
EXP_DIR = "/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps"

base_p, gap_p, valid_p, test_p = new_split_files[0], new_split_files[1], new_split_files[2], new_split_files[3]

# Tiger: base + gap
merge_and_save([base_p, gap_p], EXP_DIR, "exp_4_0.9_inter_tiger_train.json")

# 1. Exp 4.1 (Standard)
# Semantics: base + gap (Всё кроме валидации и теста)
merge_and_save([base_p, gap_p], EXP_DIR, "exp_4-1_0.9_inter_semantics_train.json")

# 2. Exp 4.2 (Short Semantics)
# Semantics: base (Короче на пропуск, без gap)
merge_and_save([base_p], EXP_DIR, "exp_4-2_0.8_inter_semantics_train.json")

# 3. Exp 4.3 (Leak)
# Semantics: base + gap + valid (Видит валидацию)
merge_and_save([base_p, gap_p, valid_p], EXP_DIR, "exp_4-3_0.95_inter_semantics_train.json")

# 4. Test Set (тест всех моделей)
merge_and_save([test_p], EXP_DIR, "test_set.json")

# 4. Valid Set (валидационный набор)
merge_and_save([valid_p], EXP_DIR, "valid_set.json")

# 4. All Set (все данные)
merge_and_save([base_p, gap_p, valid_p, test_p], EXP_DIR, "all_set.json")

print("All done!")

Merging 2 files into exp_4_0.9_inter_tiger_train.json...
✓ Done: /home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps/exp_4_0.9_inter_tiger_train.json (Users: 4016)
Merging 2 files into exp_4-1_0.9_inter_semantics_train.json...
✓ Done: /home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps/exp_4-1_0.9_inter_semantics_train.json (Users: 4016)
Merging 1 files into exp_4-2_0.8_inter_semantics_train.json...
✓ Done: /home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps/exp_4-2_0.8_inter_semantics_train.json (Users: 3813)
Merging 3 files into exp_4-3_0.95_inter_semantics_train.json...
✓ Done: /home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps/exp_4-3_0.95_inter_semantics_train.json (Users: 4118)
Merging 1 files into test_set.json...
✓ Done: /home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps/test_set.json (Users: 3154)
Merging 1 files into valid_set.json...
✓ Done: /home/jovyan/IRec/data/Yambda/updated_quant

In [1]:
with open("/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps/all_set.json", 'r') as f:
    old_inter_new = json.load(f)

with open("/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps/exp_4-1_0.9_inter_semantics_train.json", 'r') as ff:
    first_sem = json.load(ff)
    
with open("/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps/exp_4-2_0.8_inter_semantics_train.json", 'r') as ff:
    second_sem = json.load(ff)
    
with open("/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps/exp_4-3_0.95_inter_semantics_train.json", 'r') as ff:
    third_sem = json.load(ff)
    
with open("/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps/exp_4_0.9_inter_tiger_train.json", 'r') as ff:
    tiger_sem = json.load(ff)

with open("/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps/test_set.json", 'r') as ff:
    test_sem = json.load(ff)

with open("/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps/all_set.json", 'r') as ff:
    all_test_data = json.load(ff)

def check_prefix_match(full_data, subset_data, check_suffix=False):
    """
    check_suffix=True включит режим проверки суффиксов (для теста).
    """
    mismatch_count = 0
    full_match_count = 0

    num_events_full_data = sum(len(items) for items in full_data.values())
    num_events_subset_data = sum(len(items) for items in subset_data.values())
    print(f"доля событий всего {(num_events_subset_data/num_events_full_data):.2f}:")
    
    for user, sub_items in subset_data.items():
        
        if user not in full_data:
            print(f"⚠ Юзер {user} не найден в исходном файле!")
            mismatch_count += 1
            continue
            
        full_items = full_data[user]
        
        if not check_suffix:
            if len(sub_items) > len(full_items):
                mismatch_count += 1
                continue
                
            if full_items[:len(sub_items)] == sub_items:
                if len(full_items) == len(sub_items):
                    full_match_count += 1
            else:
                mismatch_count += 1

        else:
            if len(sub_items) > len(full_items):
                mismatch_count += 1
                continue

            if full_items[-len(sub_items):] == sub_items:
                 if len(full_items) == len(sub_items):
                    full_match_count += 1
            else:
                mismatch_count += 1

    mode = "СУФФИКСЫ" if check_suffix else "ПРЕФИКСЫ"
    
    if mismatch_count == 0:
        print(f"OK [{mode}] Все {len(subset_data)} массивов ОК. Полных совпадений: {full_match_count}")
    else:
        print(f"NOT OK [{mode}] Найдено {mismatch_count} ошибок.")

# --- Запуск проверок ---
print("Проверка Train сетов (должны быть префиксами):")
check_prefix_match(old_inter_new, first_sem)
check_prefix_match(old_inter_new, second_sem)
check_prefix_match(old_inter_new, third_sem)
check_prefix_match(old_inter_new, tiger_sem)

print("\nПроверка Test сета (должен быть суффиксом):")
check_prefix_match(old_inter_new, test_sem, check_suffix=True)

print("\n(Контроль) Проверка Test сета как префикса (должна упасть):")
check_prefix_match(old_inter_new, test_sem, check_suffix=False)

check_prefix_match(old_inter_new, all_test_data)


FileNotFoundError: [Errno 2] No such file or directory: '/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps/all_set.json'

In [37]:
def check_non_empty_splits(full_data, splits_data, split_names, min_history_len=2):
    """
    Проверяет, что ни одна часть истории пользователя НЕ пустая во всех разбиениях.
    """
    print("\n" + "="*80)
    print("ПРОВЕРКА НА ПУСТЫЕ ЧАСТИ ИСТОРИЙ")
    print("="*80)
    
    all_users = set(full_data.keys())
    total_issues = 0
    
    for i in range(len(split_names)):
        split_name = split_names[i]
        split_data = splits_data[i]
        print(f"\n[{split_name}] Анализ...")
        
        split_users = set(split_data.keys())
        empty_sessions = []
        
        for user, items in split_data.items():
            if not items or len(items) < min_history_len:
                empty_sessions.append(user)
        
        issues_count = len(empty_sessions)
        total_issues += issues_count
        
        print(f"  Юзеров в сплите: {len(split_users):,} / {len(all_users):,}")
        print(f"  ПУСТЫХ сессий: {len(empty_sessions)}")
        print(f"  ОБЩИХ ПРОБЛЕМ: {issues_count}")
    
    if total_issues == 0:
        print("\nВСЕ РАЗБИЕНИЯ БЕЗ ПУСТЫХ СЕССИЙ")

split_names = ['exp_4-1_0.9', 'exp_4-2_0.8', 'exp_4-3_0.95', 'exp_4_0.9_tiger', 'test_set']
splits_list = [first_sem, second_sem, third_sem, tiger_sem, test_sem]

check_non_empty_splits(old_inter_new, splits_list, split_names)



ПРОВЕРКА НА ПУСТЫЕ ЧАСТИ ИСТОРИЙ

[exp_4-1_0.9] Анализ...
  Юзеров в сплите: 4,016 / 4,138
  ПУСТЫХ сессий: 15
  ОБЩИХ ПРОБЛЕМ: 15

[exp_4-2_0.8] Анализ...
  Юзеров в сплите: 3,813 / 4,138
  ПУСТЫХ сессий: 22
  ОБЩИХ ПРОБЛЕМ: 22

[exp_4-3_0.95] Анализ...
  Юзеров в сплите: 4,118 / 4,138
  ПУСТЫХ сессий: 7
  ОБЩИХ ПРОБЛЕМ: 7

[exp_4_0.9_tiger] Анализ...
  Юзеров в сплите: 4,016 / 4,138
  ПУСТЫХ сессий: 15
  ОБЩИХ ПРОБЛЕМ: 15

[test_set] Анализ...
  Юзеров в сплите: 3,154 / 4,138
  ПУСТЫХ сессий: 105
  ОБЩИХ ПРОБЛЕМ: 105


In [27]:
EXP_DIR_FILTERED = "/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps_filtered"

base_p, gap_p, valid_p, test_p = new_split_files[0], new_split_files[1], new_split_files[2], new_split_files[3]

# Tiger: base + gap
merge_and_save_with_filter([base_p, gap_p], EXP_DIR_FILTERED, "exp_4_0.9_inter_tiger_train.json", min_history_len=2)

# 1. Exp 4.1 (Standard)
# Semantics: base + gap (Всё кроме валидации и теста)
merge_and_save_with_filter([base_p, gap_p], EXP_DIR_FILTERED, "exp_4-1_0.9_inter_semantics_train.json", min_history_len=2)

# 2. Exp 4.2 (Short Semantics)
# Semantics: base (Короче на пропуск, без gap)
merge_and_save_with_filter([base_p], EXP_DIR_FILTERED, "exp_4-2_0.8_inter_semantics_train.json", min_history_len=2)

# 3. Exp 4.3 (Leak)
# Semantics: base + gap + valid (Видит валидацию)
merge_and_save_with_filter([base_p, gap_p, valid_p], EXP_DIR_FILTERED, "exp_4-3_0.95_inter_semantics_train.json", min_history_len=2)

# 4. Test Set (тест всех моделей)
merge_and_save([test_p], EXP_DIR_FILTERED, "test_set.json")

# 4. Valid Set (валидационный набор)
merge_and_save([valid_p], EXP_DIR_FILTERED, "valid_set.json")

# 4. All Set (все данные)
merge_and_save([base_p, gap_p, valid_p, test_p], EXP_DIR_FILTERED, "all_set.json")

print("All done!")

Merging 2 files into exp_4_0.9_inter_tiger_train.json (min len=2)...
Filtered 15 users with history < 2
Remaining: 4001 users
✓ Done: /home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps_filtered/exp_4_0.9_inter_tiger_train.json (Users: 4001)
Merging 2 files into exp_4-1_0.9_inter_semantics_train.json (min len=2)...
Filtered 15 users with history < 2
Remaining: 4001 users
✓ Done: /home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps_filtered/exp_4-1_0.9_inter_semantics_train.json (Users: 4001)
Merging 1 files into exp_4-2_0.8_inter_semantics_train.json (min len=2)...
Filtered 22 users with history < 2
Remaining: 3791 users
✓ Done: /home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps_filtered/exp_4-2_0.8_inter_semantics_train.json (Users: 3791)
Merging 3 files into exp_4-3_0.95_inter_semantics_train.json (min len=2)...
Filtered 7 users with history < 2
Remaining: 4111 users
✓ Done: /home/jovyan/IRec/data/Yambda/updated_quantile_splits/me

In [ ]:
with open("/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_expsx/exp_4-1_0.9_inter_semantics_train.json", 'r') as ff:
    filtered_first_sem = json.load(ff)
    
with open("/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps_filtered/exp_4-2_0.8_inter_semantics_train.json", 'r') as ff:
    filtered_second_sem = json.load(ff)
    
with open("/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps_filtered/exp_4-3_0.95_inter_semantics_train.json", 'r') as ff:
    filtered_third_sem = json.load(ff)
    
with open("/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps_filtered/exp_4_0.9_inter_tiger_train.json", 'r') as ff:
    filtered_tiger_sem = json.load(ff)

with open("/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps_filtered/valid_set.json", 'r') as ff:
    fiiltered_valid_sem = json.load(ff)

with open("/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps_filtered/test_set.json", 'r') as ff:
    fiiltered_test_sem = json.load(ff)

with open("/home/jovyan/IRec/data/Yambda/updated_quantile_splits/merged_for_exps_filtered/all_set.json", 'r') as ff:
    filtered_all_test_data = json.load(ff)

# --- Запуск проверок ---
print("Проверка Train сетов (должны быть префиксами):")
check_prefix_match(filtered_all_test_data, filtered_first_sem)
check_prefix_match(filtered_all_test_data, filtered_second_sem)
check_prefix_match(filtered_all_test_data, filtered_third_sem)
check_prefix_match(filtered_all_test_data, filtered_tiger_sem)

print("\nПроверка Test сета (должен быть суффиксом):")
check_prefix_match(filtered_all_test_data, test_sem, check_suffix=True)

print("\n(Контроль) Проверка Test сета как префикса (должна упасть):")
check_prefix_match(filtered_all_test_data, test_sem, check_suffix=False)

check_prefix_match(filtered_all_test_data, all_test_data)

split_names = ['exp_4-1_0.9', 'exp_4-2_0.8', 'exp_4-3_0.95', 'exp_4_0.9_tiger']
splits_list_filtered = [filtered_first_sem, filtered_second_sem, filtered_third_sem, filtered_tiger_sem]

check_non_empty_splits(filtered_all_test_data, splits_list_filtered, split_names, min_history_len = 2)



Проверка Train сетов (должны быть префиксами):
доля событий всего 0.90:
✅ [ПРЕФИКСЫ] Все 4001 массивов ОК. Полных совпадений: 564
доля событий всего 0.80:
✅ [ПРЕФИКСЫ] Все 3791 массивов ОК. Полных совпадений: 343
доля событий всего 0.95:
✅ [ПРЕФИКСЫ] Все 4111 массивов ОК. Полных совпадений: 984
доля событий всего 0.90:
✅ [ПРЕФИКСЫ] Все 4001 массивов ОК. Полных совпадений: 564

Проверка Test сета (должен быть суффиксом):
доля событий всего 0.05:
✅ [СУФФИКСЫ] Все 3154 массивов ОК. Полных совпадений: 20

(Контроль) Проверка Test сета как префикса (должна упасть):
доля событий всего 0.05:
❌ [ПРЕФИКСЫ] Найдено 3134 ошибок.
доля событий всего 1.00:
✅ [ПРЕФИКСЫ] Все 4138 массивов ОК. Полных совпадений: 4138

ПРОВЕРКА НА ПУСТЫЕ ЧАСТИ ИСТОРИЙ

[exp_4-1_0.9] Анализ...
  Юзеров в сплите: 4,001 / 4,138
  ПУСТЫХ сессий: 0
  ОБЩИХ ПРОБЛЕМ: 0

[exp_4-2_0.8] Анализ...
  Юзеров в сплите: 3,791 / 4,138
  ПУСТЫХ сессий: 0
  ОБЩИХ ПРОБЛЕМ: 0

[exp_4-3_0.95] Анализ...
  Юзеров в сплите: 4,111 / 4,138
  ПУС

In [32]:
print("для теста и валидации (может упасть и скорее всего упадет)")
vt_split_names = ['valid', 'test']
vt_splits_list_filtered = [fiiltered_valid_sem, test_sem]

check_non_empty_splits(filtered_all_test_data, vt_splits_list_filtered, vt_split_names, min_history_len = 2)

для теста и валидации (может упасть и скорее всего упадет)

ПРОВЕРКА НА ПУСТЫЕ ЧАСТИ ИСТОРИЙ

[valid] Анализ...
  Юзеров в сплите: 3,120 / 4,138
  ПУСТЫХ сессий: 88
  ОБЩИХ ПРОБЛЕМ: 88

[test] Анализ...
  Юзеров в сплите: 3,154 / 4,138
  ПУСТЫХ сессий: 105
  ОБЩИХ ПРОБЛЕМ: 105


# Разбиение YAMBDA по неделям

In [7]:
global_max_time = df.select(
    pl.col("timestamps").explode().max()
).item()

# 3. Размер окна (неделя)
days_val = 1
window_sec = days_val * 24 * 3600 

# 4. Три отсечки с конца
cutoff_test_start = global_max_time - window_sec      # T - 1w
cutoff_val_start  = global_max_time - 2 * window_sec  # T - 2w
cutoff_gap_start  = global_max_time - 3 * window_sec  # T - 3w

cutoffs = [
    int(cutoff_gap_start),  # Граница Part 0 | Part 1
    int(cutoff_val_start),  # Граница Part 1 | Part 2
    int(cutoff_test_start)  # Граница Part 2 | Part 3
]

print(f"Cutoffs: {cutoffs}")

split_files = split_session_by_timestamps(
    df, 
    cutoffs, 
    output_dir="/home/jovyan/IRec/data/Yambda/day-splits/raw"
)

names = ["Base", "day -3", "day -2", "day -1"]
for i, d in enumerate(split_files):
    print(f"Part {i} [{names[i]}]: {len(d)} users")

Cutoffs: [25740785, 25827185, 25913585]
✓ Сохранено: /home/jovyan/IRec/data/Yambda/day-splits/raw/inter_new_[0_25740785).json
✓ Сохранено: /home/jovyan/IRec/data/Yambda/day-splits/raw/inter_new_[25740785_25827185).json
✓ Сохранено: /home/jovyan/IRec/data/Yambda/day-splits/raw/inter_new_[25827185_25913585).json
✓ Сохранено: /home/jovyan/IRec/data/Yambda/day-splits/raw/inter_new_[25913585_inf).json
Part 0 [Base]: 4133 users
Part 1 [day -3]: 1381 users
Part 2 [day -2]: 1350 users
Part 3 [day -1]: 1403 users


In [13]:
EXP_DIR_FILTERED = "/home/jovyan/IRec/data/Yambda/day-splits/merged_for_exps_filtered"

base_p, gap_p, valid_p, test_p = split_files[0], split_files[1], split_files[2], split_files[3]

# Tiger: base + gap
merge_and_save_with_filter([base_p, gap_p], EXP_DIR_FILTERED, "exp_4_0.9_inter_tiger_train.json", min_history_len=2)

# 1. Exp 4.1 (Standard)
# Semantics: base + gap (Всё кроме валидации и теста)
merge_and_save_with_filter([base_p, gap_p], EXP_DIR_FILTERED, "exp_4-1_0.9_inter_semantics_train.json", min_history_len=2)

# 2. Exp 4.2 (Short Semantics)
# Semantics: base (Короче на пропуск, без gap)
merge_and_save_with_filter([base_p], EXP_DIR_FILTERED, "exp_4-2_0.8_inter_semantics_train.json", min_history_len=2)

# 3. Exp 4.3 (Leak)
# Semantics: base + gap + valid (Видит валидацию)
merge_and_save_with_filter([base_p, gap_p, valid_p], EXP_DIR_FILTERED, "exp_4-3_0.95_inter_semantics_train.json", min_history_len=2)

# 4. Test Set (тест всех моделей)
merge_and_save_with_filter([test_p], EXP_DIR_FILTERED, "test_set.json", min_history_len=1)

# 4. Valid Set (валидационный набор)
merge_and_save_with_filter([valid_p], EXP_DIR_FILTERED, "valid_set.json", min_history_len=1)

# 4. All Set (все данные)
merge_and_save([base_p, gap_p, valid_p, test_p], EXP_DIR_FILTERED, "all_set.json")

print("All done!")

Merging 2 files into exp_4_0.9_inter_tiger_train.json (min len=2)...
Filtered 3 users with history < 2
Remaining: 4133 users
✓ Done: /home/jovyan/IRec/data/Yambda/day-splits/merged_for_exps_filtered/exp_4_0.9_inter_tiger_train.json (Users: 4133)
Merging 2 files into exp_4-1_0.9_inter_semantics_train.json (min len=2)...
Filtered 3 users with history < 2
Remaining: 4133 users
✓ Done: /home/jovyan/IRec/data/Yambda/day-splits/merged_for_exps_filtered/exp_4-1_0.9_inter_semantics_train.json (Users: 4133)
Merging 1 files into exp_4-2_0.8_inter_semantics_train.json (min len=2)...
Filtered 3 users with history < 2
Remaining: 4130 users
✓ Done: /home/jovyan/IRec/data/Yambda/day-splits/merged_for_exps_filtered/exp_4-2_0.8_inter_semantics_train.json (Users: 4130)
Merging 3 files into exp_4-3_0.95_inter_semantics_train.json (min len=2)...
Filtered 3 users with history < 2
Remaining: 4133 users
✓ Done: /home/jovyan/IRec/data/Yambda/day-splits/merged_for_exps_filtered/exp_4-3_0.95_inter_semantics_tra

In [14]:
def check_non_empty_splits(full_data, splits_data, split_names, min_history_len=2):
    """
    Проверяет, что ни одна часть истории пользователя НЕ пустая во всех разбиениях.
    """
    print("\n" + "="*80)
    print("ПРОВЕРКА НА ПУСТЫЕ ЧАСТИ ИСТОРИЙ")
    print("="*80)
    
    all_users = set(full_data.keys())
    total_issues = 0
    
    for i in range(len(split_names)):
        split_name = split_names[i]
        split_data = splits_data[i]
        print(f"\n[{split_name}] Анализ...")
        
        split_users = set(split_data.keys())
        empty_sessions = []
        
        for user, items in split_data.items():
            if not items or len(items) < min_history_len:
                empty_sessions.append(user)
        
        issues_count = len(empty_sessions)
        total_issues += issues_count
        
        print(f"  Юзеров в сплите: {len(split_users):,} / {len(all_users):,}")
        print(f"  ПУСТЫХ сессий: {len(empty_sessions)}")
        print(f"  ОБЩИХ ПРОБЛЕМ: {issues_count}")
    
    if total_issues == 0:
        print("\nВСЕ РАЗБИЕНИЯ БЕЗ ПУСТЫХ СЕССИЙ")

def check_prefix_match(full_data, subset_data, check_suffix=False):
    """
    check_suffix=True включит режим проверки суффиксов (для теста).
    """
    mismatch_count = 0
    full_match_count = 0
    
    # Итерируемся по ключам сабсета, так как в full_data может быть больше юзеров
    for user, sub_items in subset_data.items():
        
        # Проверяем есть ли такой юзер в исходнике
        if user not in full_data:
            print(f"⚠ Юзер {user} не найден в исходном файле!")
            mismatch_count += 1
            continue
            
        full_items = full_data[user]
        
        # Логика для проверки ПРЕФИКСА (начало совпадает)
        if not check_suffix:
            if len(sub_items) > len(full_items):
                mismatch_count += 1
                continue
                
            # Сравниваем начало full с sub
            if full_items[:len(sub_items)] == sub_items:
                if len(full_items) == len(sub_items):
                    full_match_count += 1
            else:
                mismatch_count += 1
                
        # Логика для проверки СУФФИКСА (конец совпадает - для теста)
        else:
            if len(sub_items) > len(full_items):
                mismatch_count += 1
                continue
                
            # Сравниваем конец full с sub
            # Срез [-len:] берет последние N элементов
            if full_items[-len(sub_items):] == sub_items:
                 if len(full_items) == len(sub_items):
                    full_match_count += 1
            else:
                mismatch_count += 1

    mode = "СУФФИКСЫ" if check_suffix else "ПРЕФИКСЫ"
    
    if mismatch_count == 0:
        print(f"✅ [{mode}] Все {len(subset_data)} массивов ОК. Полных совпадений: {full_match_count}")
    else:
        print(f"❌ [{mode}] Найдено {mismatch_count} ошибок.")


In [15]:
with open(f"{EXP_DIR_FILTERED}/exp_4-1_0.9_inter_semantics_train.json", 'r') as ff:
    filtered_first_sem = json.load(ff)
    
with open(f"{EXP_DIR_FILTERED}/exp_4-2_0.8_inter_semantics_train.json", 'r') as ff:
    filtered_second_sem = json.load(ff)
    
with open(f"{EXP_DIR_FILTERED}/exp_4-3_0.95_inter_semantics_train.json", 'r') as ff:
    filtered_third_sem = json.load(ff)
    
with open(f"{EXP_DIR_FILTERED}/exp_4_0.9_inter_tiger_train.json", 'r') as ff:
    filtered_tiger_sem = json.load(ff)

with open(f"{EXP_DIR_FILTERED}/valid_set.json", 'r') as ff:
    fiiltered_valid_sem = json.load(ff)

with open(f"{EXP_DIR_FILTERED}/test_set.json", 'r') as ff:
    filtered_test_sem = json.load(ff)

with open(f"{EXP_DIR_FILTERED}/all_set.json", 'r') as ff:
    filtered_all_test_data = json.load(ff)

# --- Запуск проверок ---
print("Проверка Train сетов (должны быть префиксами):")
check_prefix_match(filtered_all_test_data, filtered_first_sem)
check_prefix_match(filtered_all_test_data, filtered_second_sem)
check_prefix_match(filtered_all_test_data, filtered_third_sem)
check_prefix_match(filtered_all_test_data, filtered_tiger_sem)

print("\nПроверка Test сета (должен быть суффиксом):")
check_prefix_match(filtered_all_test_data, filtered_test_sem, check_suffix=True)

print("\n(Контроль) Проверка Test сета как префикса (должна упасть):")
check_prefix_match(filtered_all_test_data, filtered_test_sem, check_suffix=False)

check_prefix_match(filtered_all_test_data, filtered_all_test_data)


Проверка Train сетов (должны быть префиксами):
✅ [ПРЕФИКСЫ] Все 4133 массивов ОК. Полных совпадений: 2272
✅ [ПРЕФИКСЫ] Все 4130 массивов ОК. Полных совпадений: 1969
✅ [ПРЕФИКСЫ] Все 4133 массивов ОК. Полных совпадений: 2735
✅ [ПРЕФИКСЫ] Все 4133 массивов ОК. Полных совпадений: 2272

Проверка Test сета (должен быть суффиксом):
✅ [СУФФИКСЫ] Все 1403 массивов ОК. Полных совпадений: 2

(Контроль) Проверка Test сета как префикса (должна упасть):
❌ [ПРЕФИКСЫ] Найдено 1401 ошибок.
✅ [ПРЕФИКСЫ] Все 4138 массивов ОК. Полных совпадений: 4138


In [16]:
split_names = ['exp_4-1_0.9', 'exp_4-2_0.8', 'exp_4-3_0.95', 'exp_4_0.9_tiger']
splits_list_filtered = [filtered_first_sem, filtered_second_sem, filtered_third_sem, filtered_tiger_sem]

check_non_empty_splits(filtered_all_test_data, splits_list_filtered, split_names, min_history_len = 2)


ПРОВЕРКА НА ПУСТЫЕ ЧАСТИ ИСТОРИЙ

[exp_4-1_0.9] Анализ...
  Юзеров в сплите: 4,133 / 4,138
  ПУСТЫХ сессий: 0
  ОБЩИХ ПРОБЛЕМ: 0

[exp_4-2_0.8] Анализ...
  Юзеров в сплите: 4,130 / 4,138
  ПУСТЫХ сессий: 0
  ОБЩИХ ПРОБЛЕМ: 0

[exp_4-3_0.95] Анализ...
  Юзеров в сплите: 4,133 / 4,138
  ПУСТЫХ сессий: 0
  ОБЩИХ ПРОБЛЕМ: 0

[exp_4_0.9_tiger] Анализ...
  Юзеров в сплите: 4,133 / 4,138
  ПУСТЫХ сессий: 0
  ОБЩИХ ПРОБЛЕМ: 0

ВСЕ РАЗБИЕНИЯ БЕЗ ПУСТЫХ СЕССИЙ


In [17]:
filtered_all_test_data.keys()
for i, d in enumerate(split_files):
    num_users = len(d)
    
    num_events = sum(len(items) for items in d.values())
    
    print(f"{i:<10} {names[i]:<10} {num_users:<10} {num_events:<10}")

0          Base       4133       7264231   
1          day -3     1381       36676     
2          day -2     1350       35128     
3          day -1     1403       35955     
